<a href="https://colab.research.google.com/github/beingAnujChaudhary/DSFS-Joel-Grus/blob/main/notebooks/chapter_01_introduction/experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/DSFS-Joel-Grus.git

# Move into repository
%cd /content/DSFS-Joel-Grus

# Move into Chapter 1 folder
%cd notebooks/chapter_01_introduction

Mounted at /content/drive
Cloning into 'DSFS-Joel-Grus'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 99 (delta 60), reused 58 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (99/99), 834.03 KiB | 1.39 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/DSFS-Joel-Grus
/content/DSFS-Joel-Grus/notebooks/chapter_01_introduction


# Chapter 1 — Experiments

## Purpose

This notebook is your playground for experimenting with ideas introduced in Chapter 1.

The goal is to:
- Explore concepts actively
- Build intuition
- Modify code freely
- Ask “what if?” questions
- Develop first-principles understanding

i.e., Explore the data beyond the book's examples. Test assumptions, optimize parameters, and combine different signals to gain deeper insights.


**Rule:**  
> Do not worry about breaking things. Experimentation is the point.


**Mindset**:
> "What happens if I change this?" "Can I make this better?"

In [8]:
# Cell 1: Setup & Data Reuse (Self-Contained)
from collections import Counter, defaultdict

# Re-declare chapter data
users = [
    {"id": 0, "name": "Hero"}, {"id": 1, "name": "Dunn"},
    {"id": 2, "name": "Sue"},  {"id": 3, "name": "Chi"},
    {"id": 4, "name": "Thor"}, {"id": 5, "name": "Clive"},
    {"id": 6, "name": "Hicks"}, {"id": 7, "name": "Devin"},
    {"id": 8, "name": "Kate"}, {"id": 9, "name": "Klein"}
]

friendships = [
    (0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4),
    (4, 5), (5, 6), (5, 7), (6, 8), (7, 8), (8, 9)
]

for user in users:
    user["friends"] = []
for i, j in friendships:
    users[i]["friends"].append(users[j])
    users[j]["friends"].append(users[i])

interests = [
    (0, "Hadoop"), (0, "Big Data"), (0, "HBase"), (0, "Java"),
    (0, "Spark"), (0, "Storm"), (0, "Cassandra"),
    (1, "NoSQL"), (1, "MongoDB"), (1, "Cassandra"), (1, "HBase"),
    (1, "Postgres"), (2, "Python"), (2, "scikit-learn"), (2, "scipy"),
    (2, "numpy"), (2, "statsmodels"), (2, "pandas"), (3, "R"), (3, "Python"),
    (3, "statistics"), (3, "regression"), (3, "probability"),
    (4, "machine learning"), (4, "regression"), (4, "decision trees"),
    (4, "libsvm"), (5, "Python"), (5, "R"), (5, "Java"), (5, "C++"),
    (5, "Haskell"), (5, "programming languages"), (6, "statistics"),
    (6, "probability"), (6, "mathematics"), (6, "theory"),
    (7, "machine learning"), (7, "scikit-learn"), (7, "Mahout"),
    (7, "neural networks"), (8, "neural networks"), (8, "deep learning"),
    (8, "Big Data"), (8, "artificial intelligence"), (9, "Hadoop"),
    (9, "Java"), (9, "MapReduce"), (9, "Big Data")
]

# Paid/Unpaid data from the book (Experience, Status)
experience_paid_status = [
    (0.7, "paid"), (1.9, "unpaid"), (2.5, "paid"), (4.2, "unpaid"),
    (6.0, "unpaid"), (6.5, "unpaid"), (7.5, "unpaid"), (8.1, "unpaid"),
    (8.7, "paid"), (10.0, "paid")
]

print("✅ Experiment environment ready")

✅ Experiment environment ready


## Experiment 1: Degree vs. FOAF Centrality

**Hypothesis**: The book notes that Thor (id 4) has fewer friends (2) than Dunn (id 1, 3 friends), yet intuitively Thor feels more central because he bridges two groups.
**Experiment**: Calculate "Friends-of-Friends" (FOAF) counts for every user. Does FOAF count correlate better with our intuition of "importance" than raw degree?

In [ ]:
# Cell 2: Experiment 1 Code
def not_the_same(user, other_user):
    return user["id"] != other_user["id"]

def not_friends(user, other_user):
    return all(not_the_same(friend, other_user) for friend in user["friends"])

def friends_of_friend_ids_count(user):
    """Returns the count of unique friends-of-friends (excluding self and friends)."""
    foaf_ids = {
        foaf["id"]
        for friend in user["friends"]
        for foaf in friend["friends"]
        if not_the_same(user, foaf) and not_friends(user, foaf)
    }
    return len(foaf_ids)

# Calculate metrics for all users
metrics = []
for user in users:
    degree = len(user["friends"])
    foaf = friends_of_friend_ids_count(user)
    metrics.append({"id": user["id"], "name": user["name"], "degree": degree, "foaf": foaf})

# Sort by FOAF count descending
metrics_sorted_by_foaf = sorted(metrics, key=lambda x: x["foaf"], reverse=True)

print(f"{'Name':<10} | {'Degree':<6} | {'FOAF':<4}")
print("-" * 25)
for m in metrics_sorted_by_foaf:
    print(f"{m['name']:<10} | {m['degree']:<6} | {m['foaf']:<4}")

# Observation: Who ranks highest in FOAF? Does it match intuition better than Degree?

## Experiment 2: Optimizing Classification Thresholds

**Hypothesis**: The book suggests a rule: `paid` if `experience < 3.0` or `experience >= 8.5`.
**Experiment**: This was a guess. Let's use a grid search to find the *optimal* thresholds (t1, t2) that minimize misclassification errors on our existing data.
**Rule**: Predict "paid" if `exp < t1` OR `exp >= t2`, else "unpaid".

In [ ]:
# Cell 3: Experiment 2 Code
def predict_with_thresholds(exp, t1, t2):
    if exp < t1 or exp >= t2:
        return "paid"
    return "unpaid"

def find_best_thresholds(data, step=0.1):
    best_error = float('inf')
    best_params = (0, 0)

    # Search space: t1 from 0.5 to 5.0, t2 from 5.0 to 12.0
    t1_range = [round(x * step, 1) for x in range(5, 50)]
    t2_range = [round(x * step, 1) for x in range(50, 120)]

    for t1 in t1_range:
        for t2 in t2_range:
            if t1 >= t2: continue # Invalid logic

            errors = sum(1 for exp, actual in data if predict_with_thresholds(exp, t1, t2) != actual)

            if errors < best_error:
                best_error = errors
                best_params = (t1, t2)

    return best_params, best_error

optimal_t1, optimal_t2, min_error = find_best_thresholds(experience_paid_status)

print(f"Original Rule (3.0, 8.5) Error: {sum(1 for exp, actual in experience_paid_status if predict_with_thresholds(exp, 3.0, 8.5) != actual)}")
print(f"Optimal Rule ({optimal_t1}, {optimal_t2}) Error: {min_error}")

## Experiment 3: Hybrid "Data Scientists You Should Know"

**Hypothesis**: Suggesting connections based *only* on mutual friends is limited. Suggesting based *only* on interests is also limited.
**Experiment**: Create a hybrid score.
- Score += 2 for every Mutual Friend.
- Score += 1 for every Shared Interest.
- Suggest users with the highest combined score.

In [ ]:
# Cell 4: Experiment 3 Code
# Rebuild interest indexes
user_ids_by_interest = defaultdict(list)
interests_by_user_id = defaultdict(list)
for uid, interest in interests:
    user_ids_by_interest[interest].append(uid)
    interests_by_user_id[uid].append(interest)

def hybrid_suggestions(user, friend_weight=2.0, interest_weight=1.0):
    scores = defaultdict(float)

    # 1. Score Mutual Friends
    for friend in user["friends"]:
        for foaf in friend["friends"]:
            if not_the_same(user, foaf) and not_friends(user, foaf):
                scores[foaf["id"]] += friend_weight

    # 2. Score Shared Interests
    for interest in interests_by_user_id[user["id"]]:
        for other_uid in user_ids_by_interest[interest]:
            if other_uid != user["id"] and not_friends(user, other_uid):
                scores[other_uid] += interest_weight

    # Filter out existing friends
    friend_ids = {f["id"] for f in user["friends"]} | {user["id"]}
    return sorted(
        [(uid, score) for uid, score in scores.items() if uid not in friend_ids],
        key=lambda x: x[1],
        reverse=True
    )

hero = users[0]
print(f"Hybrid Suggestions for {hero['name']}:")
for uid, score in hybrid_suggestions(hero):
    print(f"  User {uid}: Score {score:.1f}")

## Experiment 4: Noise Reduction in Word Counts

**Observation**: The book's word count includes short words and splits "scikit-learn" into "scikit" and "learn".
**Experiment**: Filter out "noise" words (length < 4) to see if the top topics become more relevant.

In [ ]:
# Cell 5: Experiment 4 Code
import re

def clean_word_counts(interests_data, min_length=4):
    counts = Counter()
    for _, interest in interests_data:
        # Split by non-alphanumeric characters to handle hyphens/spaces better
        words = re.findall(r'[a-z]+', interest.lower())
        for word in words:
            if len(word) >= min_length:
                counts[word] += 1
    return counts

original_counts = Counter(word for user, interest in interests for word in interest.lower().split())
clean_counts = clean_word_counts(interests)

print("Top 5 Original Words:")
print(original_counts.most_common(5))

print("\nTop 5 Cleaned Words (len >= 4):")
print(clean_counts.most_common(5))

## Experiment 6 — What are the components of data science?

Try:

- removing one skill
- adding "Communication"
- changing order

Ask:

> Which skill feels strongest for me right now?

In [9]:
# Represent data science as components

components = [
    "Programming",
    "Mathematics",
    "Statistics",
    "Domain Knowledge"
]

print("Core components of Data Science:\n")

for component in components:
    print(f"- {component}")

Core components of Data Science:

- Programming
- Mathematics
- Statistics
- Domain Knowledge


## Experiment 7 — Rating my current strengths

Modify scores after every few chapters.

Ask:
> Which area needs most improvement?

In [10]:
# Self-assessment experiment

skills = {
    "Programming": 6,
    "Mathematics": 4,
    "Statistics": 3,
    "Domain Knowledge": 2
}

print("Current self-assessment:\n")

for skill, score in skills.items():
    print(f"{skill}: {score}/10")

Current self-assessment:

Programming: 6/10
Mathematics: 4/10
Statistics: 3/10
Domain Knowledge: 2/10


## Experiment 8 — Data science pipeline mental model

Try:

- reordering steps
- adding missing steps
- removing steps

Ask:

> What happens if data cleaning is skipped?

In [11]:
# Simple mental model of data science

pipeline = [
    "Collect Data",
    "Clean Data",
    "Analyse Data",
    "Build Model",
    "Evaluate Results",
    "Make Decisions"
]

print("Data Science Workflow:\n")

for step_number, step in enumerate(pipeline, start=1):
    print(f"{step_number}. {step}")

Data Science Workflow:

1. Collect Data
2. Clean Data
3. Analyse Data
4. Build Model
5. Evaluate Results
6. Make Decisions


## Experiment 9 — Why first-principles learning matters

Ask yourself:

> Which learning style supports my long-term AI/ML engineering goals?

In [12]:
# Compare approaches

learning_approach = {
    "Library-first": [
        "Fast",
        "Easy to start",
        "Weak intuition"
    ],

    "First-principles": [
        "Slower",
        "Deeper understanding",
        "Better debugging"
    ]
}

for approach, benefits in learning_approach.items():
    print(f"\n{approach}")

    for benefit in benefits:
        print(f" - {benefit}")


Library-first
 - Fast
 - Easy to start
 - Weak intuition

First-principles
 - Slower
 - Deeper understanding
 - Better debugging


## Experiment 10 — Mini reflection

Answer these questions honestly:

1. Which skill is currently my weakest?
2. Which skill feels strongest?
3. What surprised me about Chapter 1?
4. Why am I learning from first principles?
5. How does this connect to production AI/ML engineering?

## 📝 Reflections

**What did you learn?**
*   **Centrality**: Did Thor rank higher with FOAF? (Answer: Likely yes, as he bridges clusters).
*   **Thresholds**: What was the optimal split? (e.g., maybe 3.5 and 8.0?). This shows that data-driven rules often beat "eyeballed" rules.
*   **Hybrid**: Did combining signals give better recommendations?
*   **Noise**: Did filtering short words remove "fluff" like "big" or "java" (if short)?



## Key takeaway

Chapter 1 is not about heavy coding.

It is about building the right mindset.

The biggest lesson:

> Understanding concepts deeply matters more than using libraries quickly.

## Why this notebook matters

Most learners skip experimentation.

They only:

```txt
Read → copy code → move on
```

Bad for retention.

Your workflow becomes:

```txt
Read
↓
Implement
↓
Experiment
↓
Reflect
↓
Internalise
```

That is significantly better for long-term AI/ML engineering mastery.

